In [3]:
import pandas as pd
import numpy as np

# Load dataset baru
df = pd.read_csv(
    "../data/raw/merged_1_datetime.csv",
    parse_dates=["DATETIME"],
    decimal=","
)

print("Shape:", df.shape)

print("\nKolom:")
print(df.columns.tolist())

print("\nTipe data:")
print(df.dtypes)

print("\nPeriode data:")
print("Awal :", df["DATETIME"].min())
print("Akhir:", df["DATETIME"].max())

Shape: (324988, 12)

Kolom:
['DATETIME', 'NO2', 'SO2', 'PM25', 'PM10', 'CO', 'O3', 'TMP', 'RAIN', 'HUM', 'WS', 'WD']

Tipe data:
DATETIME    datetime64[us]
NO2                float64
SO2                float64
PM25               float64
PM10               float64
CO                 float64
O3                 float64
TMP                float64
RAIN               float64
HUM                float64
WS                 float64
WD                   int64
dtype: object

Periode data:
Awal : 2026-07-01 00:00:01
Akhir: 2026-09-21 17:40:02


In [4]:
print("Jumlah missing value:")
print(df.isna().sum())

print("\nPersentase missing value (%):")
print((df.isna().mean() * 100).round(2))

Jumlah missing value:
DATETIME    0
NO2         0
SO2         0
PM25        0
PM10        0
CO          0
O3          0
TMP         0
RAIN        0
HUM         0
WS          0
WD          0
dtype: int64

Persentase missing value (%):
DATETIME    0.0
NO2         0.0
SO2         0.0
PM25        0.0
PM10        0.0
CO          0.0
O3          0.0
TMP         0.0
RAIN        0.0
HUM         0.0
WS          0.0
WD          0.0
dtype: float64


In [5]:
print("Jumlah baris duplikat penuh:", df.duplicated().sum())

print("Jumlah DATETIME duplikat:", df["DATETIME"].duplicated().sum())

print("\nJumlah timestamp unik:", df["DATETIME"].nunique())

Jumlah baris duplikat penuh: 0
Jumlah DATETIME duplikat: 278655

Jumlah timestamp unik: 46333


In [6]:
# Urutkan berdasarkan waktu
df_sorted = df.sort_values("DATETIME").copy()

# Hitung selisih antar timestamp
time_diff = df_sorted["DATETIME"].diff().dt.total_seconds()

print("Distribusi interval waktu (detik):")
print(time_diff.value_counts().sort_index().head(20))

print("\nStatistik interval waktu (detik):")
print(time_diff.describe())

Distribusi interval waktu (detik):
DATETIME
0.0      278655
1.0       22586
2.0         181
3.0          13
4.0           3
5.0           2
7.0           1
8.0           2
9.0           1
10.0          1
11.0          1
13.0          5
16.0          1
19.0          1
55.0          1
90.0          1
209.0         1
244.0         1
280.0         1
283.0         1
Name: count, dtype: int64

Statistik interval waktu (detik):
count    324987.000000
mean         21.995960
std          82.439771
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       13500.000000
Name: DATETIME, dtype: float64


In [7]:
# Ambil timestamp unik saja
unique_times = (
    df["DATETIME"]
    .drop_duplicates()
    .sort_values()
)

# Hitung interval antar timestamp unik
unique_diff = unique_times.diff().dt.total_seconds()

print("Jumlah timestamp unik:", len(unique_times))

print("\nDistribusi interval antar timestamp unik (detik):")
print(
    unique_diff
    .value_counts()
    .sort_index()
    .head(30)
)

print("\nStatistik interval antar timestamp unik:")
print(unique_diff.describe())

Jumlah timestamp unik: 46333

Distribusi interval antar timestamp unik (detik):
DATETIME
1.0      22586
2.0        181
3.0         13
4.0          3
5.0          2
7.0          1
8.0          2
9.0          1
10.0         1
11.0         1
13.0         5
16.0         1
19.0         1
55.0         1
90.0         1
209.0        1
244.0        1
280.0        1
283.0        1
285.0        1
286.0        2
287.0        2
289.0        4
290.0        1
293.0        1
294.0        2
295.0        3
296.0       12
297.0       41
298.0     1752
Name: count, dtype: int64

Statistik interval antar timestamp unik:
count    46332.000000
mean       154.286476
std        165.109867
min          1.000000
25%          1.000000
50%        298.000000
75%        299.000000
max      13500.000000
Name: DATETIME, dtype: float64


In [8]:
# Identifikasi gap lebih dari 10 menit
long_gaps = (
    pd.DataFrame({
        "DATETIME": unique_times,
        "interval_detik": unique_diff
    })
    .query("interval_detik > 600")
    .sort_values("interval_detik", ascending=False)
)

print("Jumlah gap > 10 menit:", len(long_gaps))

print("\nGap terbesar:")
print(long_gaps.head(20).to_string(index=False))

Jumlah gap > 10 menit: 30

Gap terbesar:
           DATETIME  interval_detik
2026-08-05 19:30:02         13500.0
2026-09-16 15:45:01          2099.0
2026-07-02 07:40:01          1799.0
2026-09-16 14:45:02          1200.0
2026-09-16 16:40:02          1200.0
2026-09-16 17:00:02          1200.0
2026-09-09 08:35:01          1199.0
2026-09-16 14:05:01          1199.0
2026-07-23 18:10:01          1199.0
2026-09-16 15:10:01          1199.0
2026-09-16 14:25:01          1199.0
2026-09-16 16:05:01          1198.0
2026-07-06 17:40:03           901.0
2026-07-06 22:35:02           900.0
2026-07-01 18:30:02           900.0
2026-07-02 03:30:02           900.0
2026-07-06 02:30:02           900.0
2026-07-31 09:40:02           900.0
2026-07-18 11:30:02           900.0
2026-07-06 23:15:02           900.0


In [9]:
pollutant_cols = [
    "PM25",
    "PM10",
    "CO",
    "O3",
    "SO2",
    "NO2"
]

print(
    df[pollutant_cols]
    .describe()
    .T
)

         count        mean         std    min     25%     50%     75%      max
PM25  324988.0   33.625496   41.734136    1.2   14.46   22.03   34.87   802.74
PM10  324988.0   69.183940   80.192892    3.2   31.67   48.30   75.53  1945.77
CO    324988.0  507.076645  400.609252  135.7  386.52  420.55  480.37  6638.45
O3    324988.0   22.849636   24.232887    0.0    2.22   13.92   41.36   149.18
SO2   324988.0    2.117839    5.255009    0.0    0.00    0.00    3.72   191.76
NO2   324988.0   23.176849   19.423142    0.0    3.05   21.38   39.52   179.86


In [10]:
print("Jumlah nilai <= 0:")
print(
    (df[pollutant_cols] <= 0)
    .sum()
)

print("\nJumlah nilai negatif:")
print(
    (df[pollutant_cols] < 0)
    .sum()
)

print("\nPersentase nilai nol (%):")
print(
    (df[pollutant_cols] == 0)
    .mean()
    .mul(100)
    .round(2)
)

Jumlah nilai <= 0:
PM25         0
PM10         0
CO           0
O3       68435
SO2     172404
NO2      69215
dtype: int64

Jumlah nilai negatif:
PM25    0
PM10    0
CO      0
O3      0
SO2     0
NO2     0
dtype: int64

Persentase nilai nol (%):
PM25     0.00
PM10     0.00
CO       0.00
O3      21.06
SO2     53.05
NO2     21.30
dtype: float64


In [11]:
zero_summary = pd.DataFrame({
    "Jumlah_0": (df[pollutant_cols] == 0).sum(),
    "Persentase_0": (
        (df[pollutant_cols] == 0)
        .mean()
        .mul(100)
        .round(2)
    )
})

print(zero_summary)

      Jumlah_0  Persentase_0
PM25         0          0.00
PM10         0          0.00
CO           0          0.00
O3       68435         21.06
SO2     172404         53.05
NO2      69215         21.30


In [12]:
outlier_summary = []

for col in pollutant_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()
    
    outlier_summary.append({
        "Polutan": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Batas_Bawah": lower_bound,
        "Batas_Atas": upper_bound,
        "Jumlah_Outlier": outlier_count,
        "Persentase_Outlier": outlier_count / len(df) * 100
    })

outlier_summary = pd.DataFrame(outlier_summary)

print(outlier_summary.round(2).to_string(index=False))

Polutan     Q1     Q3   IQR  Batas_Bawah  Batas_Atas  Jumlah_Outlier  Persentase_Outlier
   PM25  14.46  34.87 20.41       -16.15       65.48           32476                9.99
   PM10  31.67  75.53 43.86       -34.12      141.32           28003                8.62
     CO 386.52 480.37 93.85       245.74      621.14           37252               11.46
     O3   2.22  41.36 39.14       -56.49      100.07             668                0.21
    SO2   0.00   3.72  3.72        -5.58        9.30            5536                1.70
    NO2   3.05  39.52 36.47       -51.66       94.23              16                0.00


In [13]:
correlation_cols = [
    "PM25",
    "PM10",
    "CO",
    "O3",
    "SO2",
    "NO2",
    "TMP",
    "RAIN",
    "HUM",
    "WS"
]

correlation_matrix = df[correlation_cols].corr()

print(correlation_matrix.round(2))

      PM25  PM10    CO    O3   SO2   NO2   TMP  RAIN   HUM    WS
PM25  1.00  0.99  0.90 -0.18  0.45  0.05 -0.25 -0.01  0.23 -0.07
PM10  0.99  1.00  0.89 -0.17  0.45  0.04 -0.27 -0.00  0.25 -0.08
CO    0.90  0.89  1.00 -0.14  0.61  0.09 -0.17 -0.01  0.15 -0.04
O3   -0.18 -0.17 -0.14  1.00 -0.12 -0.38  0.45 -0.01 -0.47  0.13
SO2   0.45  0.45  0.61 -0.12  1.00  0.08 -0.09 -0.00  0.08 -0.01
NO2   0.05  0.04  0.09 -0.38  0.08  1.00 -0.37  0.01  0.38 -0.11
TMP  -0.25 -0.27 -0.17  0.45 -0.09 -0.37  1.00 -0.04 -0.95  0.23
RAIN -0.01 -0.00 -0.01 -0.01 -0.00  0.01 -0.04  1.00  0.04  0.04
HUM   0.23  0.25  0.15 -0.47  0.08  0.38 -0.95  0.04  1.00 -0.25
WS   -0.07 -0.08 -0.04  0.13 -0.01 -0.11  0.23  0.04 -0.25  1.00


In [14]:
daily_pollutants = (
    df.set_index("DATETIME")[pollutant_cols]
    .resample("1D")
    .mean()
)

print("Shape:", daily_pollutants.shape)

print("\n5 hari pertama:")
print(daily_pollutants.head())

print("\n5 hari terakhir:")
print(daily_pollutants.tail())

Shape: (83, 6)

5 hari pertama:
                 PM25       PM10          CO         O3       SO2        NO2
DATETIME                                                                    
2026-07-01  16.160220  34.485260  397.706682  16.414211  1.716011  21.445655
2026-07-02  20.219159  43.371189  436.468168  15.137737  1.664909  21.833539
2026-07-03  20.444248  42.744235  420.991357  17.487744  1.857058  22.494613
2026-07-04  24.111208  51.280881  444.418482  16.587615  1.868841  23.900891
2026-07-05  19.400532  42.845797  420.693999  19.172959  1.866360  22.335895

5 hari terakhir:
                 PM25        PM10          CO         O3       SO2        NO2
DATETIME                                                                     
2026-09-17  72.512314  139.891715  846.350402  17.093475  2.970655  23.664555
2026-09-18  54.999711  108.373586  733.606098  21.822517  4.447935  25.688108
2026-09-19  65.433409  126.480015  853.700391  23.793068  4.619701  25.347507
2026-09-20  53.461165

In [15]:
df["DATETIME_5MIN"] = df["DATETIME"].dt.floor("5min")

print("Jumlah timestamp unik sebelum:", df["DATETIME"].nunique())
print("Jumlah timestamp unik sesudah:", df["DATETIME_5MIN"].nunique())

print("\nContoh:")
print(
    df[["DATETIME", "DATETIME_5MIN"]]
    .head(10)
)

Jumlah timestamp unik sebelum: 46333
Jumlah timestamp unik sesudah: 23533

Contoh:
             DATETIME DATETIME_5MIN
0 2026-07-01 00:00:01    2026-07-01
1 2026-07-01 00:00:02    2026-07-01
2 2026-07-01 00:00:02    2026-07-01
3 2026-07-01 00:00:02    2026-07-01
4 2026-07-01 00:00:02    2026-07-01
5 2026-07-01 00:00:02    2026-07-01
6 2026-07-01 00:00:02    2026-07-01
7 2026-07-01 00:00:02    2026-07-01
8 2026-07-01 00:00:02    2026-07-01
9 2026-07-01 00:00:02    2026-07-01


In [16]:
df_5min = (
    df.groupby("DATETIME_5MIN")
    .agg({
        "NO2": "mean",
        "SO2": "mean",
        "PM25": "mean",
        "PM10": "mean",
        "CO": "mean",
        "O3": "mean",
        "TMP": "mean",
        "RAIN": "mean",
        "HUM": "mean",
        "WS": "mean",
        "WD": "mean"
    })
    .reset_index()
)

print("Shape:", df_5min.shape)

print("\nKolom:")
print(df_5min.columns.tolist())

print("\nMissing value:")
print(df_5min.isna().sum())

Shape: (23533, 12)

Kolom:
['DATETIME_5MIN', 'NO2', 'SO2', 'PM25', 'PM10', 'CO', 'O3', 'TMP', 'RAIN', 'HUM', 'WS', 'WD']

Missing value:
DATETIME_5MIN    0
NO2              0
SO2              0
PM25             0
PM10             0
CO               0
O3               0
TMP              0
RAIN             0
HUM              0
WS               0
WD               0
dtype: int64


In [17]:
expected_5min = pd.date_range(
    start=df_5min["DATETIME_5MIN"].min(),
    end=df_5min["DATETIME_5MIN"].max(),
    freq="5min"
)

actual_5min = pd.DatetimeIndex(df_5min["DATETIME_5MIN"])

missing_5min = expected_5min.difference(actual_5min)

print("Periode:", df_5min["DATETIME_5MIN"].min(), "→", df_5min["DATETIME_5MIN"].max())
print("Jumlah interval 5-menit yang seharusnya:", len(expected_5min))
print("Jumlah interval 5-menit tersedia:", len(actual_5min))
print("Jumlah interval 5-menit hilang:", len(missing_5min))

print("\nContoh interval yang hilang:")
print(missing_5min[:20])

Periode: 2026-07-01 00:00:00 → 2026-09-21 17:40:00
Jumlah interval 5-menit yang seharusnya: 23829
Jumlah interval 5-menit tersedia: 23533
Jumlah interval 5-menit hilang: 296

Contoh interval yang hilang:
DatetimeIndex(['2026-07-01 00:15:00', '2026-07-01 00:40:00',
               '2026-07-01 04:45:00', '2026-07-01 04:55:00',
               '2026-07-01 05:10:00', '2026-07-01 05:55:00',
               '2026-07-01 06:10:00', '2026-07-01 06:30:00',
               '2026-07-01 06:35:00', '2026-07-01 07:00:00',
               '2026-07-01 07:05:00', '2026-07-01 10:00:00',
               '2026-07-01 17:20:00', '2026-07-01 17:30:00',
               '2026-07-01 18:00:00', '2026-07-01 18:20:00',
               '2026-07-01 18:25:00', '2026-07-01 19:20:00',
               '2026-07-01 19:35:00', '2026-07-01 23:50:00'],
              dtype='datetime64[us]', freq=None)


In [18]:
# Buat full timeline 5-menit
full_5min = pd.date_range(
    start=df_5min["DATETIME_5MIN"].min(),
    end=df_5min["DATETIME_5MIN"].max(),
    freq="5min"
)

actual_5min = pd.DatetimeIndex(df_5min["DATETIME_5MIN"])

# Cari interval yang hilang
missing_5min = full_5min.difference(actual_5min)

# Hitung panjang setiap gap berturut-turut
missing_series = pd.Series(missing_5min)

gap_group = (
    missing_series.diff() != pd.Timedelta(minutes=5)
).cumsum()

gap_lengths = (
    missing_series
    .groupby(gap_group)
    .size()
)

print("Jumlah kelompok gap:", len(gap_lengths))

print("\nDistribusi panjang gap:")
print(gap_lengths.value_counts().sort_index())

print("\nGap terpanjang (dalam jumlah interval 5-menit):")
print(gap_lengths.sort_values(ascending=False).head(20))

Jumlah kelompok gap: 212

Distribusi panjang gap:
1     186
2      14
3       9
5       1
6       1
44      1
Name: count, dtype: int64

Gap terpanjang (dalam jumlah interval 5-menit):
97     44
205     6
26      5
82      3
206     3
167     3
202     3
204     3
201     3
208     3
209     3
203     3
42      2
51      2
52      2
46      2
21      2
23      2
9       2
14      2
dtype: int64


In [19]:
# ============================================================
# Membuat timeline 5-menit lengkap
# dan interpolasi hanya untuk gap <= 15 menit
# ============================================================

# Kolom numerik yang akan diinterpolasi
numeric_cols = [
    "NO2", "SO2", "PM25", "PM10", "CO", "O3",
    "TMP", "RAIN", "HUM", "WS", "WD"
]

# Jadikan datetime sebagai index
df_5min_full = (
    df_5min
    .set_index("DATETIME_5MIN")
    .reindex(full_5min)
)

# Hitung jumlah missing sebelum interpolasi
missing_before = df_5min_full[numeric_cols].isna().sum()

# Interpolasi hanya gap pendek (maksimal 3 interval = 15 menit)
df_5min_full[numeric_cols] = (
    df_5min_full[numeric_cols]
    .interpolate(
        method="time",
        limit=3,
        limit_area="inside"
    )
)

# Hitung jumlah missing setelah interpolasi
missing_after = df_5min_full[numeric_cols].isna().sum()

print("Missing sebelum interpolasi:")
print(missing_before)

print("\nMissing setelah interpolasi:")
print(missing_after)

print("\nShape setelah membuat timeline lengkap:", df_5min_full.shape)

Missing sebelum interpolasi:
NO2     296
SO2     296
PM25    296
PM10    296
CO      296
O3      296
TMP     296
RAIN    296
HUM     296
WS      296
WD      296
dtype: int64

Missing setelah interpolasi:
NO2     46
SO2     46
PM25    46
PM10    46
CO      46
O3      46
TMP     46
RAIN    46
HUM     46
WS      46
WD      46
dtype: int64

Shape setelah membuat timeline lengkap: (23829, 11)


In [20]:
# ============================================================
# Agregasi data 5-menit menjadi data hourly
# ============================================================

df_hourly = (
    df_5min_full
    .resample("1h")
    .agg({
        "NO2": "mean",
        "SO2": "mean",
        "PM25": "mean",
        "PM10": "mean",
        "CO": "mean",
        "O3": "mean",
        "TMP": "mean",
        "RAIN": "mean",
        "HUM": "mean",
        "WS": "mean",
        "WD": "mean"
    })
)

print("Shape:", df_hourly.shape)

print("\nPeriode:")
print(df_hourly.index.min(), "→", df_hourly.index.max())

print("\nMissing values:")
print(df_hourly.isna().sum())

print("\n5 baris pertama:")
display(df_hourly.head())

print("\n5 baris terakhir:")
display(df_hourly.tail())

Shape: (1986, 11)

Periode:
2026-07-01 00:00:00 → 2026-09-21 17:00:00

Missing values:
NO2     2
SO2     2
PM25    2
PM10    2
CO      2
O3      2
TMP     2
RAIN    2
HUM     2
WS      2
WD      2
dtype: int64

5 baris pertama:


,NO2,SO2,PM25,PM10,CO,O3,TMP,RAIN,HUM,WS,WD
2026-07-01 00:00:00,17.096474,1.569536,21.255265,43.898704,465.064224,21.457800,31.756087,0.0,73.586702,1.463298,141.083861
2026-07-01 01:00:00,20.026875,1.206058,23.089072,47.641170,448.993323,21.626055,30.734312,0.0,76.818837,1.423762,138.170814
2026-07-01 02:00:00,23.489118,1.292720,18.582015,40.324981,409.513921,19.704114,29.748591,0.0,78.965175,1.209339,140.101735
2026-07-01 03:00:00,30.437735,1.394721,15.844247,35.921184,401.504320,15.859188,29.124881,0.0,81.594595,1.079078,134.229871
2026-07-01 04:00:00,30.034439,1.212319,16.771483,38.846235,396.398652,15.006194,28.745444,0.0,83.300049,1.059880,128.262500



5 baris terakhir:


,NO2,SO2,PM25,PM10,CO,O3,TMP,RAIN,HUM,WS,WD
2026-09-21 13:00:00,27.987245,3.417179,128.356494,237.354324,1420.508745,6.462568,25.475103,0.0,90.494891,0.543133,99.868175
2026-09-21 14:00:00,30.705205,3.792791,123.154360,228.974841,1405.434579,8.674049,25.391380,0.0,91.362888,0.741977,144.189442
2026-09-21 15:00:00,35.402939,2.266170,95.276561,174.378134,1322.767480,7.769038,27.150403,0.0,87.265845,1.047237,121.729186
2026-09-21 16:00:00,25.300885,1.966328,76.864691,136.562966,1002.958754,20.197229,30.886950,0.0,76.875790,1.291652,169.174853
2026-09-21 17:00:00,11.193537,1.771957,55.423392,99.176302,639.042579,38.541698,34.011652,0.0,66.491052,1.391176,125.736048


In [21]:
# Cari timestamp hourly yang seluruh variabelnya missing

missing_hours = df_hourly.index[
    df_hourly.isna().all(axis=1)
]

print("Jumlah jam yang seluruh datanya missing:", len(missing_hours))

print("\nTimestamp yang missing:")
for ts in missing_hours:
    print(ts)

Jumlah jam yang seluruh datanya missing: 2

Timestamp yang missing:
2026-08-05 17:00:00
2026-08-05 18:00:00


In [23]:
# ============================================================
# Cek kontinuitas timeline hourly
# ============================================================

expected_hourly = pd.date_range(
    start=df_hourly.index.min(),
    end=df_hourly.index.max(),
    freq="1h"
)

actual_hourly = pd.DatetimeIndex(df_hourly.index)

missing_timestamps_hourly = expected_hourly.difference(actual_hourly)

print("Jumlah jam yang seharusnya:", len(expected_hourly))
print("Jumlah jam dalam df_hourly:", len(actual_hourly))
print("Jumlah timestamp hourly yang hilang dari index:", len(missing_timestamps_hourly))

if len(missing_timestamps_hourly) > 0:
    print("\nTimestamp yang hilang dari index:")
    print(missing_timestamps_hourly)
else:
    print("\nTidak ada timestamp yang hilang dari index.")

Jumlah jam yang seharusnya: 1986
Jumlah jam dalam df_hourly: 1986
Jumlah timestamp hourly yang hilang dari index: 0

Tidak ada timestamp yang hilang dari index.


In [24]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

pollutant_cols = [
    "PM25", "PM10", "CO", "O3", "SO2", "NO2"
]

# Salin dataset hourly
df_model = df_hourly.copy()

# ------------------------------------------------------------
# 1. Lag features untuk masing-masing polutan
# ------------------------------------------------------------

lag_hours = [1, 2, 3, 6, 24]

for col in pollutant_cols:
    for lag in lag_hours:
        df_model[f"{col}_lag_{lag}"] = df_model[col].shift(lag)

# ------------------------------------------------------------
# 2. Rolling mean backward-looking
# ------------------------------------------------------------

rolling_windows = [3, 6]

for col in pollutant_cols:
    for window in rolling_windows:
        df_model[f"{col}_roll_mean_{window}"] = (
            df_model[col]
            .rolling(window=window)
            .mean()
        )

# ------------------------------------------------------------
# 3. Wind Direction → sin/cos
# ------------------------------------------------------------

df_model["WD_sin"] = np.sin(np.deg2rad(df_model["WD"]))
df_model["WD_cos"] = np.cos(np.deg2rad(df_model["WD"]))

# ------------------------------------------------------------
# 4. Fitur waktu
# ------------------------------------------------------------

df_model["hour"] = df_model.index.hour
df_model["day_of_week"] = df_model.index.dayofweek

df_model["hour_sin"] = np.sin(
    2 * np.pi * df_model["hour"] / 24
)

df_model["hour_cos"] = np.cos(
    2 * np.pi * df_model["hour"] / 24
)

print("Shape df_model:", df_model.shape)

print("\nJumlah kolom:", len(df_model.columns))

print("\nKolom:")
print(df_model.columns.tolist())

Shape df_model: (1986, 59)

Jumlah kolom: 59

Kolom:
['NO2', 'SO2', 'PM25', 'PM10', 'CO', 'O3', 'TMP', 'RAIN', 'HUM', 'WS', 'WD', 'PM25_lag_1', 'PM25_lag_2', 'PM25_lag_3', 'PM25_lag_6', 'PM25_lag_24', 'PM10_lag_1', 'PM10_lag_2', 'PM10_lag_3', 'PM10_lag_6', 'PM10_lag_24', 'CO_lag_1', 'CO_lag_2', 'CO_lag_3', 'CO_lag_6', 'CO_lag_24', 'O3_lag_1', 'O3_lag_2', 'O3_lag_3', 'O3_lag_6', 'O3_lag_24', 'SO2_lag_1', 'SO2_lag_2', 'SO2_lag_3', 'SO2_lag_6', 'SO2_lag_24', 'NO2_lag_1', 'NO2_lag_2', 'NO2_lag_3', 'NO2_lag_6', 'NO2_lag_24', 'PM25_roll_mean_3', 'PM25_roll_mean_6', 'PM10_roll_mean_3', 'PM10_roll_mean_6', 'CO_roll_mean_3', 'CO_roll_mean_6', 'O3_roll_mean_3', 'O3_roll_mean_6', 'SO2_roll_mean_3', 'SO2_roll_mean_6', 'NO2_roll_mean_3', 'NO2_roll_mean_6', 'WD_sin', 'WD_cos', 'hour', 'day_of_week', 'hour_sin', 'hour_cos']


In [25]:
# ============================================================
# TARGET FORECASTING PM25
# ============================================================

forecast_horizons = [1, 2, 3, 4, 5, 6, 7, 72]

for h in forecast_horizons:
    df_model[f"PM25_target_{h}h"] = df_model["PM25"].shift(-h)

print("Target PM25 yang dibuat:")

target_cols = [
    f"PM25_target_{h}h"
    for h in forecast_horizons
]

print(target_cols)

print("\nContoh target:")
display(
    df_model[
        ["PM25"] + target_cols
    ].head(10)
)

Target PM25 yang dibuat:
['PM25_target_1h', 'PM25_target_2h', 'PM25_target_3h', 'PM25_target_4h', 'PM25_target_5h', 'PM25_target_6h', 'PM25_target_7h', 'PM25_target_72h']

Contoh target:


,PM25,PM25_target_1h,PM25_target_2h,PM25_target_3h,PM25_target_4h,PM25_target_5h,PM25_target_6h,PM25_target_7h,PM25_target_72h
2026-07-01 00:00:00,21.255265,23.089072,18.582015,15.844247,16.771483,17.262692,16.236874,15.043346,11.679967
2026-07-01 01:00:00,23.089072,18.582015,15.844247,16.771483,17.262692,16.236874,15.043346,13.030133,10.978287
2026-07-01 02:00:00,18.582015,15.844247,16.771483,17.262692,16.236874,15.043346,13.030133,19.579156,20.787002
2026-07-01 03:00:00,15.844247,16.771483,17.262692,16.236874,15.043346,13.030133,19.579156,15.741405,20.303581
2026-07-01 04:00:00,16.771483,17.262692,16.236874,15.043346,13.030133,19.579156,15.741405,13.285045,23.624347
2026-07-01 05:00:00,17.262692,16.236874,15.043346,13.030133,19.579156,15.741405,13.285045,14.867859,24.954108
2026-07-01 06:00:00,16.236874,15.043346,13.030133,19.579156,15.741405,13.285045,14.867859,15.676932,31.444314
2026-07-01 07:00:00,15.043346,13.030133,19.579156,15.741405,13.285045,14.867859,15.676932,19.978202,39.485187
2026-07-01 08:00:00,13.030133,19.579156,15.741405,13.285045,14.867859,15.676932,19.978202,20.677318,42.228276
2026-07-01 09:00:00,19.579156,15.741405,13.285045,14.867859,15.676932,19.978202,20.677318,14.707075,37.775815


In [26]:
# ============================================================
# TARGET FORECASTING UNTUK SEMUA POLUTAN
# ============================================================

for col in pollutant_cols:
    for h in forecast_horizons:
        target_col = f"{col}_target_{h}h"
        
        # PM25 sudah dibuat sebelumnya, jadi tidak perlu dibuat ulang
        if target_col not in df_model.columns:
            df_model[target_col] = df_model[col].shift(-h)

# Daftar seluruh target
all_target_cols = [
    f"{col}_target_{h}h"
    for col in pollutant_cols
    for h in forecast_horizons
]

print("Jumlah target:", len(all_target_cols))

print("\nDaftar target:")
print(all_target_cols)

print("\nShape df_model:", df_model.shape)

Jumlah target: 48

Daftar target:
['PM25_target_1h', 'PM25_target_2h', 'PM25_target_3h', 'PM25_target_4h', 'PM25_target_5h', 'PM25_target_6h', 'PM25_target_7h', 'PM25_target_72h', 'PM10_target_1h', 'PM10_target_2h', 'PM10_target_3h', 'PM10_target_4h', 'PM10_target_5h', 'PM10_target_6h', 'PM10_target_7h', 'PM10_target_72h', 'CO_target_1h', 'CO_target_2h', 'CO_target_3h', 'CO_target_4h', 'CO_target_5h', 'CO_target_6h', 'CO_target_7h', 'CO_target_72h', 'O3_target_1h', 'O3_target_2h', 'O3_target_3h', 'O3_target_4h', 'O3_target_5h', 'O3_target_6h', 'O3_target_7h', 'O3_target_72h', 'SO2_target_1h', 'SO2_target_2h', 'SO2_target_3h', 'SO2_target_4h', 'SO2_target_5h', 'SO2_target_6h', 'SO2_target_7h', 'SO2_target_72h', 'NO2_target_1h', 'NO2_target_2h', 'NO2_target_3h', 'NO2_target_4h', 'NO2_target_5h', 'NO2_target_6h', 'NO2_target_7h', 'NO2_target_72h']

Shape df_model: (1986, 107)


In [28]:
# ============================================================
# CEK DATA VALID UNTUK SETIAP TARGET
# ============================================================

target_validity = []

for target_col in all_target_cols:
    valid_count = df_model[target_col].notna().sum()
    missing_count = df_model[target_col].isna().sum()
    
    target_validity.append({
        "target": target_col,
        "valid": valid_count,
        "missing": missing_count,
        "valid_pct": valid_count / len(df_model) * 100
    })

target_validity = pd.DataFrame(target_validity)

display(target_validity)

print("\nRingkasan berdasarkan horizon:")

target_validity["horizon"] = (
    target_validity["target"]
    .str.extract(r"_(\d+)h$")[0]
    .astype(int)
)

display(
    target_validity
    .groupby("horizon")[["valid", "missing", "valid_pct"]]
    .first()
)

,target,valid,missing,valid_pct
0,PM25_target_1h,1983,3,99.848943
1,PM25_target_2h,1982,4,99.798590
2,PM25_target_3h,1981,5,99.748238
3,PM25_target_4h,1980,6,99.697885
4,PM25_target_5h,1979,7,99.647533
5,PM25_target_6h,1978,8,99.597180
6,PM25_target_7h,1977,9,99.546828
7,PM25_target_72h,1912,74,96.273917
8,PM10_target_1h,1983,3,99.848943
9,PM10_target_2h,1982,4,99.798590



Ringkasan berdasarkan horizon:


,valid,missing,valid_pct
horizon,,,
1,1983,3,99.848943
2,1982,4,99.798590
3,1981,5,99.748238
4,1980,6,99.697885
5,1979,7,99.647533
6,1978,8,99.597180
7,1977,9,99.546828
72,1912,74,96.273917


In [29]:
# ============================================================
# CEK MISSING PADA FITUR
# ============================================================

# Pisahkan kolom fitur dari target
feature_cols = [
    col for col in df_model.columns
    if col not in all_target_cols
]

feature_missing = (
    df_model[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Jumlah fitur:", len(feature_cols))

print("\nFitur yang memiliki missing:")
print(feature_missing[feature_missing > 0])

print("\nTotal baris yang memiliki minimal 1 fitur missing:",
      df_model[feature_cols].isna().any(axis=1).sum())

Jumlah fitur: 59

Fitur yang memiliki missing:
CO_lag_24           26
PM10_lag_24         26
PM25_lag_24         26
SO2_lag_24          26
NO2_lag_24          26
O3_lag_24           26
PM10_roll_mean_6    12
PM25_roll_mean_6    12
CO_roll_mean_6      12
SO2_roll_mean_6     12
O3_roll_mean_6      12
NO2_roll_mean_6     12
PM25_lag_6           8
CO_lag_6             8
PM10_lag_6           8
O3_lag_6             8
SO2_lag_6            8
NO2_lag_6            8
PM10_roll_mean_3     6
PM25_roll_mean_3     6
CO_roll_mean_3       6
O3_roll_mean_3       6
SO2_roll_mean_3      6
NO2_roll_mean_3      6
CO_lag_3             5
PM25_lag_3           5
PM10_lag_3           5
SO2_lag_3            5
O3_lag_3             5
NO2_lag_3            5
CO_lag_2             4
PM25_lag_2           4
O3_lag_2             4
SO2_lag_2            4
NO2_lag_2            4
PM10_lag_2           4
NO2_lag_1            3
PM10_lag_1           3
SO2_lag_1            3
PM25_lag_1           3
CO_lag_1             3
O3_lag_1  

In [30]:
# ============================================================
# TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

n = len(df_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_end_time = df_model.index[train_end - 1]
val_end_time = df_model.index[val_end - 1]

print("Total observasi:", n)

print("\nTRAIN")
print("Index :", 0, "→", train_end - 1)
print("Jumlah:", train_end)
print("Periode:", df_model.index[0], "→", train_end_time)

print("\nVALIDATION")
print("Index :", train_end, "→", val_end - 1)
print("Jumlah:", val_end - train_end)
print("Periode:", df_model.index[train_end], "→", val_end_time)

print("\nTEST")
print("Index :", val_end, "→", n - 1)
print("Jumlah:", n - val_end)
print("Periode:", df_model.index[val_end], "→", df_model.index[-1])

Total observasi: 1986

TRAIN
Index : 0 → 1389
Jumlah: 1390
Periode: 2026-07-01 00:00:00 → 2026-08-27 21:00:00

VALIDATION
Index : 1390 → 1687
Jumlah: 298
Periode: 2026-08-27 22:00:00 → 2026-09-09 07:00:00

TEST
Index : 1688 → 1985
Jumlah: 298
Periode: 2026-09-09 08:00:00 → 2026-09-21 17:00:00


In [31]:
# ============================================================
# FINAL FEATURE LIST
# ============================================================

# Kolom yang tidak digunakan sebagai fitur model
exclude_cols = (
    all_target_cols
    + ["WD"]
)

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]

print("Jumlah fitur final:", len(feature_cols))

print("\nFitur final:")
for i, col in enumerate(feature_cols, start=1):
    print(f"{i:>2}. {col}")

Jumlah fitur final: 58

Fitur final:
 1. NO2
 2. SO2
 3. PM25
 4. PM10
 5. CO
 6. O3
 7. TMP
 8. RAIN
 9. HUM
10. WS
11. PM25_lag_1
12. PM25_lag_2
13. PM25_lag_3
14. PM25_lag_6
15. PM25_lag_24
16. PM10_lag_1
17. PM10_lag_2
18. PM10_lag_3
19. PM10_lag_6
20. PM10_lag_24
21. CO_lag_1
22. CO_lag_2
23. CO_lag_3
24. CO_lag_6
25. CO_lag_24
26. O3_lag_1
27. O3_lag_2
28. O3_lag_3
29. O3_lag_6
30. O3_lag_24
31. SO2_lag_1
32. SO2_lag_2
33. SO2_lag_3
34. SO2_lag_6
35. SO2_lag_24
36. NO2_lag_1
37. NO2_lag_2
38. NO2_lag_3
39. NO2_lag_6
40. NO2_lag_24
41. PM25_roll_mean_3
42. PM25_roll_mean_6
43. PM10_roll_mean_3
44. PM10_roll_mean_6
45. CO_roll_mean_3
46. CO_roll_mean_6
47. O3_roll_mean_3
48. O3_roll_mean_6
49. SO2_roll_mean_3
50. SO2_roll_mean_6
51. NO2_roll_mean_3
52. NO2_roll_mean_6
53. WD_sin
54. WD_cos
55. hour
56. day_of_week
57. hour_sin
58. hour_cos


In [33]:
# ============================================================
# FUNGSI MEMBANGUN DATASET EKSPERIMEN
# ============================================================

def build_dataset(target_col):
    """
    Membuat X dan y untuk satu target tertentu.
    Hanya baris dengan seluruh fitur dan target yang tersedia
    yang digunakan.
    """
    
    data = df_model[feature_cols + [target_col]].copy()
    
    # Hanya gunakan baris yang lengkap
    data = data.dropna()
    
    X = data[feature_cols]
    y = data[target_col]
    
    return X, y


# Contoh: PM25 1 jam
X_pm25_1h, y_pm25_1h = build_dataset("PM25_target_1h")

print("X shape:", X_pm25_1h.shape)
print("y shape:", y_pm25_1h.shape)

print("\nPeriode data:")
print(X_pm25_1h.index.min(), "→", X_pm25_1h.index.max())

print("\nMissing X:", X_pm25_1h.isna().sum().sum())
print("Missing y:", y_pm25_1h.isna().sum())

X shape: (1950, 58)
y shape: (1950,)

Periode data:
2026-07-02 00:00:00 → 2026-09-21 16:00:00

Missing X: 0
Missing y: 0


In [34]:
# ============================================================
# FUNGSI TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

def time_split(X, y):
    """
    Membagi data berdasarkan batas waktu yang sama
    untuk seluruh eksperimen model.
    """
    
    train_mask = X.index <= train_end_time
    
    val_mask = (
        (X.index > train_end_time) &
        (X.index <= val_end_time)
    )
    
    test_mask = X.index > val_end_time
    
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
    
    X_val = X.loc[val_mask]
    y_val = y.loc[val_mask]
    
    X_test = X.loc[test_mask]
    y_test = y.loc[test_mask]
    
    return (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )


# Uji untuk PM25 1 jam
(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test
) = time_split(X_pm25_1h, y_pm25_1h)


print("TRAIN")
print("Shape X:", X_train.shape)
print("Shape y:", y_train.shape)
print("Periode:", X_train.index.min(), "→", X_train.index.max())

print("\nVALIDATION")
print("Shape X:", X_val.shape)
print("Shape y:", y_val.shape)
print("Periode:", X_val.index.min(), "→", X_val.index.max())

print("\nTEST")
print("Shape X:", X_test.shape)
print("Shape y:", y_test.shape)
print("Periode:", X_test.index.min(), "→", X_test.index.max())

TRAIN
Shape X: (1355, 58)
Shape y: (1355,)
Periode: 2026-07-02 00:00:00 → 2026-08-27 21:00:00

VALIDATION
Shape X: (298, 58)
Shape y: (298,)
Periode: 2026-08-27 22:00:00 → 2026-09-09 07:00:00

TEST
Shape X: (297, 58)
Shape y: (297,)
Periode: 2026-09-09 08:00:00 → 2026-09-21 16:00:00


In [35]:
# ============================================================
# BASELINE PERSISTENCE - PM25 1 JAM
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Persistence:
# prediksi = nilai PM25 pada waktu sekarang
y_pred_val_persistence = X_val["PM25"].values
y_pred_test_persistence = X_test["PM25"].values

# Evaluation - Validation
mae_val = mean_absolute_error(y_val, y_pred_val_persistence)
rmse_val = np.sqrt(
    mean_squared_error(y_val, y_pred_val_persistence)
)
r2_val = r2_score(y_val, y_pred_val_persistence)

# Evaluation - Test
mae_test = mean_absolute_error(y_test, y_pred_test_persistence)
rmse_test = np.sqrt(
    mean_squared_error(y_test, y_pred_test_persistence)
)
r2_test = r2_score(y_test, y_pred_test_persistence)

print("=== PERSISTENCE — PM25 1H ===")

print("\nValidation:")
print(f"MAE  : {mae_val:.4f}")
print(f"RMSE : {rmse_val:.4f}")
print(f"R²   : {r2_val:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test:.4f}")
print(f"RMSE : {rmse_test:.4f}")
print(f"R²   : {r2_test:.4f}")

=== PERSISTENCE — PM25 1H ===

Validation:
MAE  : 8.9482
RMSE : 14.6619
R²   : 0.5864

Test:
MAE  : 11.8651
RMSE : 18.5289
R²   : 0.6007


In [36]:
# ============================================================
# RIDGE — PM25 1 JAM
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

# Training hanya menggunakan TRAIN
ridge_model.fit(X_train, y_train)

# Prediction
y_pred_val_ridge = ridge_model.predict(X_val)
y_pred_test_ridge = ridge_model.predict(X_test)

# Validation
mae_val_ridge = mean_absolute_error(y_val, y_pred_val_ridge)
rmse_val_ridge = np.sqrt(
    mean_squared_error(y_val, y_pred_val_ridge)
)
r2_val_ridge = r2_score(y_val, y_pred_val_ridge)

# Test
mae_test_ridge = mean_absolute_error(y_test, y_pred_test_ridge)
rmse_test_ridge = np.sqrt(
    mean_squared_error(y_test, y_pred_test_ridge)
)
r2_test_ridge = r2_score(y_test, y_pred_test_ridge)

print("=== RIDGE — PM25 1H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_ridge:.4f}")
print(f"RMSE : {rmse_val_ridge:.4f}")
print(f"R²   : {r2_val_ridge:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_ridge:.4f}")
print(f"RMSE : {rmse_test_ridge:.4f}")
print(f"R²   : {r2_test_ridge:.4f}")

=== RIDGE — PM25 1H ===

Validation:
MAE  : 9.2399
RMSE : 15.0510
R²   : 0.5642

Test:
MAE  : 11.7385
RMSE : 17.7007
R²   : 0.6356


In [37]:
# ============================================================
# RANDOM FOREST — PM25 1 JAM
# ============================================================

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

# Training
rf_model.fit(X_train, y_train)

# Prediction
y_pred_val_rf = rf_model.predict(X_val)
y_pred_test_rf = rf_model.predict(X_test)

# Validation
mae_val_rf = mean_absolute_error(y_val, y_pred_val_rf)
rmse_val_rf = np.sqrt(
    mean_squared_error(y_val, y_pred_val_rf)
)
r2_val_rf = r2_score(y_val, y_pred_val_rf)

# Test
mae_test_rf = mean_absolute_error(y_test, y_pred_test_rf)
rmse_test_rf = np.sqrt(
    mean_squared_error(y_test, y_pred_test_rf)
)
r2_test_rf = r2_score(y_test, y_pred_test_rf)

print("=== RANDOM FOREST — PM25 1H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_rf:.4f}")
print(f"RMSE : {rmse_val_rf:.4f}")
print(f"R²   : {r2_val_rf:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_rf:.4f}")
print(f"RMSE : {rmse_test_rf:.4f}")
print(f"R²   : {r2_test_rf:.4f}")

=== RANDOM FOREST — PM25 1H ===

Validation:
MAE  : 8.8182
RMSE : 15.1193
R²   : 0.5602

Test:
MAE  : 14.1550
RMSE : 22.7839
R²   : 0.3963


In [38]:
# ============================================================
# HISTOGRAM GRADIENT BOOSTING — PM25 1 JAM
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor

hgb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

# Training
hgb_model.fit(X_train, y_train)

# Prediction
y_pred_val_hgb = hgb_model.predict(X_val)
y_pred_test_hgb = hgb_model.predict(X_test)

# Validation
mae_val_hgb = mean_absolute_error(y_val, y_pred_val_hgb)
rmse_val_hgb = np.sqrt(
    mean_squared_error(y_val, y_pred_val_hgb)
)
r2_val_hgb = r2_score(y_val, y_pred_val_hgb)

# Test
mae_test_hgb = mean_absolute_error(y_test, y_pred_test_hgb)
rmse_test_hgb = np.sqrt(
    mean_squared_error(y_test, y_pred_test_hgb)
)
r2_test_hgb = r2_score(y_test, y_pred_test_hgb)

print("=== HISTGRADIENTBOOSTING — PM25 1H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_hgb:.4f}")
print(f"RMSE : {rmse_val_hgb:.4f}")
print(f"R²   : {r2_val_hgb:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_hgb:.4f}")
print(f"RMSE : {rmse_test_hgb:.4f}")
print(f"R²   : {r2_test_hgb:.4f}")

=== HISTGRADIENTBOOSTING — PM25 1H ===

Validation:
MAE  : 8.8985
RMSE : 14.7803
R²   : 0.5797

Test:
MAE  : 14.7549
RMSE : 23.1648
R²   : 0.3760


In [39]:
# ============================================================
# DATASET EKSPERIMEN PM25 — 3 JAM
# ============================================================

X_pm25_3h, y_pm25_3h = build_dataset("PM25_target_3h")

(
    X_train_3h, y_train_3h,
    X_val_3h, y_val_3h,
    X_test_3h, y_test_3h
) = time_split(X_pm25_3h, y_pm25_3h)

print("TRAIN")
print("X:", X_train_3h.shape)
print("y:", y_train_3h.shape)

print("\nVALIDATION")
print("X:", X_val_3h.shape)
print("y:", y_val_3h.shape)

print("\nTEST")
print("X:", X_test_3h.shape)
print("y:", y_test_3h.shape)

print("\nPeriode:")
print(
    X_train_3h.index.min(), "→", X_train_3h.index.max()
)
print(
    X_val_3h.index.min(), "→", X_val_3h.index.max()
)
print(
    X_test_3h.index.min(), "→", X_test_3h.index.max()
)

TRAIN
X: (1354, 58)
y: (1354,)

VALIDATION
X: (298, 58)
y: (298,)

TEST
X: (295, 58)
y: (295,)

Periode:
2026-07-02 00:00:00 → 2026-08-27 21:00:00
2026-08-27 22:00:00 → 2026-09-09 07:00:00
2026-09-09 08:00:00 → 2026-09-21 14:00:00


In [40]:
# ============================================================
# BASELINE PERSISTENCE — PM25 3 JAM
# ============================================================

y_pred_val_persistence_3h = X_val_3h["PM25"].values
y_pred_test_persistence_3h = X_test_3h["PM25"].values

# Validation
mae_val_persistence_3h = mean_absolute_error(
    y_val_3h,
    y_pred_val_persistence_3h
)

rmse_val_persistence_3h = np.sqrt(
    mean_squared_error(
        y_val_3h,
        y_pred_val_persistence_3h
    )
)

r2_val_persistence_3h = r2_score(
    y_val_3h,
    y_pred_val_persistence_3h
)

# Test
mae_test_persistence_3h = mean_absolute_error(
    y_test_3h,
    y_pred_test_persistence_3h
)

rmse_test_persistence_3h = np.sqrt(
    mean_squared_error(
        y_test_3h,
        y_pred_test_persistence_3h
    )
)

r2_test_persistence_3h = r2_score(
    y_test_3h,
    y_pred_test_persistence_3h
)

print("=== PERSISTENCE — PM25 3H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_persistence_3h:.4f}")
print(f"RMSE : {rmse_val_persistence_3h:.4f}")
print(f"R²   : {r2_val_persistence_3h:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_persistence_3h:.4f}")
print(f"RMSE : {rmse_test_persistence_3h:.4f}")
print(f"R²   : {r2_test_persistence_3h:.4f}")

=== PERSISTENCE — PM25 3H ===

Validation:
MAE  : 16.0107
RMSE : 23.8997
R²   : -0.1061

Test:
MAE  : 21.3016
RMSE : 29.6001
R²   : -0.0160


In [41]:
# ============================================================
# RIDGE — PM25 3 JAM
# ============================================================

ridge_model_3h = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_model_3h.fit(X_train_3h, y_train_3h)

y_pred_val_ridge_3h = ridge_model_3h.predict(X_val_3h)
y_pred_test_ridge_3h = ridge_model_3h.predict(X_test_3h)

# Validation
mae_val_ridge_3h = mean_absolute_error(
    y_val_3h,
    y_pred_val_ridge_3h
)

rmse_val_ridge_3h = np.sqrt(
    mean_squared_error(
        y_val_3h,
        y_pred_val_ridge_3h
    )
)

r2_val_ridge_3h = r2_score(
    y_val_3h,
    y_pred_val_ridge_3h
)

# Test
mae_test_ridge_3h = mean_absolute_error(
    y_test_3h,
    y_pred_test_ridge_3h
)

rmse_test_ridge_3h = np.sqrt(
    mean_squared_error(
        y_test_3h,
        y_pred_test_ridge_3h
    )
)

r2_test_ridge_3h = r2_score(
    y_test_3h,
    y_pred_test_ridge_3h
)

print("=== RIDGE — PM25 3H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_ridge_3h:.4f}")
print(f"RMSE : {rmse_val_ridge_3h:.4f}")
print(f"R²   : {r2_val_ridge_3h:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_ridge_3h:.4f}")
print(f"RMSE : {rmse_test_ridge_3h:.4f}")
print(f"R²   : {r2_test_ridge_3h:.4f}")

=== RIDGE — PM25 3H ===

Validation:
MAE  : 12.7020
RMSE : 20.2944
R²   : 0.2025

Test:
MAE  : 16.9616
RMSE : 24.1383
R²   : 0.3244


In [42]:
# ============================================================
# RANDOM FOREST — PM25 3 JAM
# ============================================================

rf_model_3h = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

rf_model_3h.fit(X_train_3h, y_train_3h)

y_pred_val_rf_3h = rf_model_3h.predict(X_val_3h)
y_pred_test_rf_3h = rf_model_3h.predict(X_test_3h)

# Validation
mae_val_rf_3h = mean_absolute_error(
    y_val_3h,
    y_pred_val_rf_3h
)

rmse_val_rf_3h = np.sqrt(
    mean_squared_error(
        y_val_3h,
        y_pred_val_rf_3h
    )
)

r2_val_rf_3h = r2_score(
    y_val_3h,
    y_pred_val_rf_3h
)

# Test
mae_test_rf_3h = mean_absolute_error(
    y_test_3h,
    y_pred_test_rf_3h
)

rmse_test_rf_3h = np.sqrt(
    mean_squared_error(
        y_test_3h,
        y_pred_test_rf_3h
    )
)

r2_test_rf_3h = r2_score(
    y_test_3h,
    y_pred_test_rf_3h
)

print("=== RANDOM FOREST — PM25 3H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_rf_3h:.4f}")
print(f"RMSE : {rmse_val_rf_3h:.4f}")
print(f"R²   : {r2_val_rf_3h:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_rf_3h:.4f}")
print(f"RMSE : {rmse_test_rf_3h:.4f}")
print(f"R²   : {r2_test_rf_3h:.4f}")

=== RANDOM FOREST — PM25 3H ===

Validation:
MAE  : 12.1000
RMSE : 19.1869
R²   : 0.2871

Test:
MAE  : 18.6204
RMSE : 27.5768
R²   : 0.1182


In [43]:
# ============================================================
# HISTGRADIENTBOOSTING — PM25 3 JAM
# ============================================================

hgb_model_3h = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hgb_model_3h.fit(X_train_3h, y_train_3h)

y_pred_val_hgb_3h = hgb_model_3h.predict(X_val_3h)
y_pred_test_hgb_3h = hgb_model_3h.predict(X_test_3h)

# Validation
mae_val_hgb_3h = mean_absolute_error(
    y_val_3h,
    y_pred_val_hgb_3h
)

rmse_val_hgb_3h = np.sqrt(
    mean_squared_error(
        y_val_3h,
        y_pred_val_hgb_3h
    )
)

r2_val_hgb_3h = r2_score(
    y_val_3h,
    y_pred_val_hgb_3h
)

# Test
mae_test_hgb_3h = mean_absolute_error(
    y_test_3h,
    y_pred_test_hgb_3h
)

rmse_test_hgb_3h = np.sqrt(
    mean_squared_error(
        y_test_3h,
        y_pred_test_hgb_3h
    )
)

r2_test_hgb_3h = r2_score(
    y_test_3h,
    y_pred_test_hgb_3h
)

print("=== HISTGRADIENTBOOSTING — PM25 3H ===")

print("\nValidation:")
print(f"MAE  : {mae_val_hgb_3h:.4f}")
print(f"RMSE : {rmse_val_hgb_3h:.4f}")
print(f"R²   : {r2_val_hgb_3h:.4f}")

print("\nTest:")
print(f"MAE  : {mae_test_hgb_3h:.4f}")
print(f"RMSE : {rmse_test_hgb_3h:.4f}")
print(f"R²   : {r2_test_hgb_3h:.4f}")

=== HISTGRADIENTBOOSTING — PM25 3H ===

Validation:
MAE  : 13.1099
RMSE : 19.4924
R²   : 0.2643

Test:
MAE  : 20.3624
RMSE : 29.0972
R²   : 0.0183


In [44]:
# ============================================================
# DATASET EKSPERIMEN PM25 — 7 JAM
# ============================================================

X_pm25_7h, y_pm25_7h = build_dataset("PM25_target_7h")

(
    X_train_7h, y_train_7h,
    X_val_7h, y_val_7h,
    X_test_7h, y_test_7h
) = time_split(X_pm25_7h, y_pm25_7h)

print("TRAIN")
print("X:", X_train_7h.shape)
print("y:", y_train_7h.shape)

print("\nVALIDATION")
print("X:", X_val_7h.shape)
print("y:", y_val_7h.shape)

print("\nTEST")
print("X:", X_test_7h.shape)
print("y:", y_test_7h.shape)

print("\nPeriode:")
print(
    X_train_7h.index.min(), "→", X_train_7h.index.max()
)
print(
    X_val_7h.index.min(), "→", X_val_7h.index.max()
)
print(
    X_test_7h.index.min(), "→", X_test_7h.index.max()
)

TRAIN
X: (1354, 58)
y: (1354,)

VALIDATION
X: (298, 58)
y: (298,)

TEST
X: (291, 58)
y: (291,)

Periode:
2026-07-02 00:00:00 → 2026-08-27 21:00:00
2026-08-27 22:00:00 → 2026-09-09 07:00:00
2026-09-09 08:00:00 → 2026-09-21 10:00:00


In [45]:
# Persistence Baseline - PM25 7H

y_pred_val_persist_7h = X_val_7h["PM25"]
y_pred_test_persist_7h = X_test_7h["PM25"]

print("=== PM25 7H - Persistence Baseline ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_7h, y_pred_val_persist_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_7h, y_pred_val_persist_7h)))
print("R²  :", r2_score(y_val_7h, y_pred_val_persist_7h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_7h, y_pred_test_persist_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_7h, y_pred_test_persist_7h)))
print("R²  :", r2_score(y_test_7h, y_pred_test_persist_7h))

=== PM25 7H - Persistence Baseline ===

Validation:
MAE : 23.590300705966147
RMSE: 31.796908525869636
R²  : -0.9629636410674036

Test:
MAE : 32.028199549701625
RMSE: 41.27381556532864
R²  : -0.9676287508712471


In [46]:
# Ridge - PM25 7H

ridge_7h = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_7h.fit(X_train_7h, y_train_7h)

y_pred_val_ridge_7h = ridge_7h.predict(X_val_7h)
y_pred_test_ridge_7h = ridge_7h.predict(X_test_7h)

print("=== PM25 7H - Ridge ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_7h, y_pred_val_ridge_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_7h, y_pred_val_ridge_7h)))
print("R²  :", r2_score(y_val_7h, y_pred_val_ridge_7h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_7h, y_pred_test_ridge_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_7h, y_pred_test_ridge_7h)))
print("R²  :", r2_score(y_test_7h, y_pred_test_ridge_7h))

=== PM25 7H - Ridge ===

Validation:
MAE : 13.870261812093283
RMSE: 21.22256480774291
R²  : 0.12554352589332252

Test:
MAE : 20.489489884856905
RMSE: 28.789049463429507
R²  : 0.04269886624582264


In [47]:
# Random Forest - PM25 7H

rf_7h = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

rf_7h.fit(X_train_7h, y_train_7h)

y_pred_val_rf_7h = rf_7h.predict(X_val_7h)
y_pred_test_rf_7h = rf_7h.predict(X_test_7h)

print("=== PM25 7H - Random Forest ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_7h, y_pred_val_rf_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_7h, y_pred_val_rf_7h)))
print("R²  :", r2_score(y_val_7h, y_pred_val_rf_7h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_7h, y_pred_test_rf_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_7h, y_pred_test_rf_7h)))
print("R²  :", r2_score(y_test_7h, y_pred_test_rf_7h))

=== PM25 7H - Random Forest ===

Validation:
MAE : 13.765300816703062
RMSE: 20.66445180449362
R²  : 0.17093183779155763

Test:
MAE : 20.161092892856804
RMSE: 29.529433675060808
R²  : -0.007173191753621744


In [48]:
# HistGradientBoosting - PM25 7H

hgb_7h = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hgb_7h.fit(X_train_7h, y_train_7h)

y_pred_val_hgb_7h = hgb_7h.predict(X_val_7h)
y_pred_test_hgb_7h = hgb_7h.predict(X_test_7h)

print("=== PM25 7H - HistGradientBoosting ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_7h, y_pred_val_hgb_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_7h, y_pred_val_hgb_7h)))
print("R²  :", r2_score(y_val_7h, y_pred_val_hgb_7h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_7h, y_pred_test_hgb_7h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_7h, y_pred_test_hgb_7h)))
print("R²  :", r2_score(y_test_7h, y_pred_test_hgb_7h))

=== PM25 7H - HistGradientBoosting ===

Validation:
MAE : 14.062105010351873
RMSE: 20.90261859929096
R²  : 0.15171096565790076

Test:
MAE : 20.702304499134154
RMSE: 30.485125995590654
R²  : -0.07342056177647382


In [49]:
X_pm25_72h, y_pm25_72h = build_dataset("PM25_target_72h")

(
    X_train_72h, y_train_72h,
    X_val_72h, y_val_72h,
    X_test_72h, y_test_72h
) = time_split(X_pm25_72h, y_pm25_72h)

print("TRAIN X:", X_train_72h.shape, "y:", y_train_72h.shape)
print("VALIDATION X:", X_val_72h.shape, "y:", y_val_72h.shape)
print("TEST X:", X_test_72h.shape, "y:", y_test_72h.shape)

print("\nPeriode:")
print(X_train_72h.index.min(), "→", X_train_72h.index.max())
print(X_val_72h.index.min(), "→", X_val_72h.index.max())
print(X_test_72h.index.min(), "→", X_test_72h.index.max())

TRAIN X: (1354, 58) y: (1354,)
VALIDATION X: (298, 58) y: (298,)
TEST X: (226, 58) y: (226,)

Periode:
2026-07-02 00:00:00 → 2026-08-27 21:00:00
2026-08-27 22:00:00 → 2026-09-09 07:00:00
2026-09-09 08:00:00 → 2026-09-18 17:00:00


In [50]:
# Persistence Baseline - PM25 72H

y_pred_val_persist_72h = X_val_72h["PM25"]
y_pred_test_persist_72h = X_test_72h["PM25"]

print("=== PM25 72H - Persistence Baseline ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_72h, y_pred_val_persist_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_72h, y_pred_val_persist_72h)))
print("R²  :", r2_score(y_val_72h, y_pred_val_persist_72h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_72h, y_pred_test_persist_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_72h, y_pred_test_persist_72h)))
print("R²  :", r2_score(y_test_72h, y_pred_test_persist_72h))

=== PM25 72H - Persistence Baseline ===

Validation:
MAE : 17.095409324264583
RMSE: 24.64134935910928
R²  : -0.20428106452767492

Test:
MAE : 23.65892214863698
RMSE: 31.63781613055843
R²  : -0.141038805351702


In [51]:
# Ridge - PM25 72H

ridge_72h = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_72h.fit(X_train_72h, y_train_72h)

y_pred_val_ridge_72h = ridge_72h.predict(X_val_72h)
y_pred_test_ridge_72h = ridge_72h.predict(X_test_72h)

print("=== PM25 72H - Ridge ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_72h, y_pred_val_ridge_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_72h, y_pred_val_ridge_72h)))
print("R²  :", r2_score(y_val_72h, y_pred_val_ridge_72h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_72h, y_pred_test_ridge_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_72h, y_pred_test_ridge_72h)))
print("R²  :", r2_score(y_test_72h, y_pred_test_ridge_72h))

=== PM25 72H - Ridge ===

Validation:
MAE : 17.210527585130748
RMSE: 25.009377493287577
R²  : -0.24052251043323647

Test:
MAE : 21.3177660442806
RMSE: 29.00080180790871
R²  : 0.04124546829742792


In [52]:
# Random Forest - PM25 72H

rf_72h = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

rf_72h.fit(X_train_72h, y_train_72h)

y_pred_val_rf_72h = rf_72h.predict(X_val_72h)
y_pred_test_rf_72h = rf_72h.predict(X_test_72h)

print("=== PM25 72H - Random Forest ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_72h, y_pred_val_rf_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_72h, y_pred_val_rf_72h)))
print("R²  :", r2_score(y_val_72h, y_pred_val_rf_72h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_72h, y_pred_test_rf_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_72h, y_pred_test_rf_72h)))
print("R²  :", r2_score(y_test_72h, y_pred_test_rf_72h))

=== PM25 72H - Random Forest ===

Validation:
MAE : 14.01064008373235
RMSE: 19.748713645003853
R²  : 0.22647115925779615

Test:
MAE : 19.84339635646207
RMSE: 28.026024422523115
R²  : 0.10461376973258041


In [53]:
# HistGradientBoosting - PM25 72H

hgb_72h = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hgb_72h.fit(X_train_72h, y_train_72h)

y_pred_val_hgb_72h = hgb_72h.predict(X_val_72h)
y_pred_test_hgb_72h = hgb_72h.predict(X_test_72h)

print("=== PM25 72H - HistGradientBoosting ===")

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val_72h, y_pred_val_hgb_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_val_72h, y_pred_val_hgb_72h)))
print("R²  :", r2_score(y_val_72h, y_pred_val_hgb_72h))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test_72h, y_pred_test_hgb_72h))
print("RMSE:", np.sqrt(mean_squared_error(y_test_72h, y_pred_test_hgb_72h)))
print("R²  :", r2_score(y_test_72h, y_pred_test_hgb_72h))

=== PM25 72H - HistGradientBoosting ===

Validation:
MAE : 14.984126638895399
RMSE: 21.013857245205138
R²  : 0.12418890864677434

Test:
MAE : 21.23256023762715
RMSE: 30.216943255369817
R²  : -0.04085079212894449


In [55]:
# ============================================================
# Fungsi eksperimen otomatis untuk seluruh pollutant & horizon
# ============================================================

def run_experiment(target_col, pollutant, horizon):
    """
    Menjalankan 4 model untuk satu pollutant dan satu horizon:
    - Persistence
    - Ridge
    - Random Forest
    - HistGradientBoosting

    Pemilihan model nantinya berdasarkan validation set.
    """

    # 1. Bangun dataset
    X, y = build_dataset(target_col)

    # 2. Time-based split
    (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    ) = time_split(X, y)

    results = []

    # --------------------------------------------------------
    # Model 1: Persistence
    # --------------------------------------------------------
    y_pred_val = X_val[pollutant]
    y_pred_test = X_test[pollutant]

    results.append({
        "Pollutant": pollutant,
        "Horizon": horizon,
        "Model": "Persistence",
        "Val_MAE": mean_absolute_error(y_val, y_pred_val),
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, y_pred_val)),
        "Val_R2": r2_score(y_val, y_pred_val),
        "Test_MAE": mean_absolute_error(y_test, y_pred_test),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Test_R2": r2_score(y_test, y_pred_test)
    })

    # --------------------------------------------------------
    # Model 2: Ridge
    # --------------------------------------------------------
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    ridge.fit(X_train, y_train)

    y_pred_val = ridge.predict(X_val)
    y_pred_test = ridge.predict(X_test)

    results.append({
        "Pollutant": pollutant,
        "Horizon": horizon,
        "Model": "Ridge",
        "Val_MAE": mean_absolute_error(y_val, y_pred_val),
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, y_pred_val)),
        "Val_R2": r2_score(y_val, y_pred_val),
        "Test_MAE": mean_absolute_error(y_test, y_pred_test),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Test_R2": r2_score(y_test, y_pred_test)
    })

    # --------------------------------------------------------
    # Model 3: Random Forest
    # --------------------------------------------------------
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        max_features="sqrt"
    )

    rf.fit(X_train, y_train)

    y_pred_val = rf.predict(X_val)
    y_pred_test = rf.predict(X_test)

    results.append({
        "Pollutant": pollutant,
        "Horizon": horizon,
        "Model": "Random Forest",
        "Val_MAE": mean_absolute_error(y_val, y_pred_val),
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, y_pred_val)),
        "Val_R2": r2_score(y_val, y_pred_val),
        "Test_MAE": mean_absolute_error(y_test, y_pred_test),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Test_R2": r2_score(y_test, y_pred_test)
    })

    # --------------------------------------------------------
    # Model 4: HistGradientBoosting
    # --------------------------------------------------------
    hgb = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    )

    hgb.fit(X_train, y_train)

    y_pred_val = hgb.predict(X_val)
    y_pred_test = hgb.predict(X_test)

    results.append({
        "Pollutant": pollutant,
        "Horizon": horizon,
        "Model": "HistGradientBoosting",
        "Val_MAE": mean_absolute_error(y_val, y_pred_val),
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, y_pred_val)),
        "Val_R2": r2_score(y_val, y_pred_val),
        "Test_MAE": mean_absolute_error(y_test, y_pred_test),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Test_R2": r2_score(y_test, y_pred_test)
    })

    return pd.DataFrame(results)

In [56]:
# Test fungsi otomatis: PM25 1H

results_pm25_1h = run_experiment(
    target_col="PM25_target_1h",
    pollutant="PM25",
    horizon="1H"
)

results_pm25_1h

,Pollutant,Horizon,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
0,PM25,1H,Persistence,8.948232,14.661851,0.586404,11.865065,18.528891,0.600741
1,PM25,1H,Ridge,9.239895,15.050960,0.564160,11.738513,17.700651,0.635637
2,PM25,1H,Random Forest,8.818201,15.119264,0.560195,14.155029,22.783887,0.396313
3,PM25,1H,HistGradientBoosting,8.898540,14.780298,0.579694,14.754872,23.164775,0.375960


In [57]:
# Daftar seluruh kombinasi eksperimen

experiment_configs = []

for pollutant in pollutant_cols:
    for horizon in forecast_horizons:
        experiment_configs.append({
            "pollutant": pollutant,
            "horizon": horizon,
            "target_col": f"{pollutant}_target_{horizon}h"
        })

experiment_configs = pd.DataFrame(experiment_configs)

print("Jumlah kombinasi:", len(experiment_configs))
display(experiment_configs)

Jumlah kombinasi: 48


,pollutant,horizon,target_col
0,PM25,1,PM25_target_1h
1,PM25,2,PM25_target_2h
2,PM25,3,PM25_target_3h
3,PM25,4,PM25_target_4h
4,PM25,5,PM25_target_5h
5,PM25,6,PM25_target_6h
6,PM25,7,PM25_target_7h
7,PM25,72,PM25_target_72h
8,PM10,1,PM10_target_1h
9,PM10,2,PM10_target_2h


In [58]:
# ============================================================
# Jalankan seluruh eksperimen
# 6 pollutant × 8 horizon × 4 model = 192 model runs
# ============================================================

all_results = []

total_experiments = len(experiment_configs)

for i, config in experiment_configs.iterrows():

    pollutant = config["pollutant"]
    horizon = config["horizon"]
    target_col = config["target_col"]

    print(
        f"[{i+1}/{total_experiments}] "
        f"{pollutant} - {horizon}H",
        flush=True
    )

    result = run_experiment(
        target_col=target_col,
        pollutant=pollutant,
        horizon=f"{horizon}H"
    )

    all_results.append(result)

# Gabungkan seluruh hasil
results_all = pd.concat(
    all_results,
    ignore_index=True
)

print("\n========================================")
print("EKSPERIMEN SELESAI")
print("========================================")
print("Jumlah hasil:", len(results_all))
print("Seharusnya:", 192)

display(results_all)

[1/48] PM25 - 1H
[2/48] PM25 - 2H
[3/48] PM25 - 3H
[4/48] PM25 - 4H
[5/48] PM25 - 5H
[6/48] PM25 - 6H
[7/48] PM25 - 7H
[8/48] PM25 - 72H
[9/48] PM10 - 1H
[10/48] PM10 - 2H
[11/48] PM10 - 3H
[12/48] PM10 - 4H
[13/48] PM10 - 5H
[14/48] PM10 - 6H
[15/48] PM10 - 7H
[16/48] PM10 - 72H
[17/48] CO - 1H
[18/48] CO - 2H
[19/48] CO - 3H
[20/48] CO - 4H
[21/48] CO - 5H
[22/48] CO - 6H
[23/48] CO - 7H
[24/48] CO - 72H
[25/48] O3 - 1H
[26/48] O3 - 2H
[27/48] O3 - 3H
[28/48] O3 - 4H
[29/48] O3 - 5H
[30/48] O3 - 6H
[31/48] O3 - 7H
[32/48] O3 - 72H
[33/48] SO2 - 1H
[34/48] SO2 - 2H
[35/48] SO2 - 3H
[36/48] SO2 - 4H
[37/48] SO2 - 5H
[38/48] SO2 - 6H
[39/48] SO2 - 7H
[40/48] SO2 - 72H
[41/48] NO2 - 1H
[42/48] NO2 - 2H
[43/48] NO2 - 3H
[44/48] NO2 - 4H
[45/48] NO2 - 5H
[46/48] NO2 - 6H
[47/48] NO2 - 7H
[48/48] NO2 - 72H

EKSPERIMEN SELESAI
Jumlah hasil: 192
Seharusnya: 192


,Pollutant,Horizon,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
0,PM25,1H,Persistence,8.948232,14.661851,0.586404,11.865065,18.528891,0.600741
1,PM25,1H,Ridge,9.239895,15.050960,0.564160,11.738513,17.700651,0.635637
2,PM25,1H,Random Forest,8.818201,15.119264,0.560195,14.155029,22.783887,0.396313
3,PM25,1H,HistGradientBoosting,8.898540,14.780298,0.579694,14.754872,23.164775,0.375960
4,PM25,2H,Persistence,13.174042,20.808006,0.164106,17.607932,25.429621,0.248892
...,...,...,...,...,...,...,...,...,...
187,NO2,7H,HistGradientBoosting,2.748506,3.580098,0.833400,2.878412,3.731725,0.843291
188,NO2,72H,Persistence,3.297258,4.321231,0.759324,4.326859,5.707314,0.622149
189,NO2,72H,Ridge,4.378305,5.635968,0.590593,4.561269,5.900675,0.596112
190,NO2,72H,Random Forest,2.628617,3.465965,0.845166,3.069033,4.014975,0.813008


In [59]:
# Cek kelengkapan hasil eksperimen

print("Shape results_all:", results_all.shape)

print("\nJumlah model:")
print(results_all["Model"].value_counts())

print("\nJumlah hasil per pollutant:")
print(results_all["Pollutant"].value_counts())

print("\nJumlah hasil per horizon:")
print(results_all["Horizon"].value_counts())

print("\nJumlah nilai NaN:")
print(results_all.isna().sum())

print("\nJumlah kombinasi Pollutant-Horizon:")
print(
    results_all[["Pollutant", "Horizon"]]
    .drop_duplicates()
    .shape[0]
)

Shape results_all: (192, 9)

Jumlah model:
Model
Persistence             48
Ridge                   48
Random Forest           48
HistGradientBoosting    48
Name: count, dtype: int64

Jumlah hasil per pollutant:
Pollutant
PM25    32
PM10    32
CO      32
O3      32
SO2     32
NO2     32
Name: count, dtype: int64

Jumlah hasil per horizon:
Horizon
1H     24
2H     24
3H     24
4H     24
5H     24
6H     24
7H     24
72H    24
Name: count, dtype: int64

Jumlah nilai NaN:
Pollutant    0
Horizon      0
Model        0
Val_MAE      0
Val_RMSE     0
Val_R2       0
Test_MAE     0
Test_RMSE    0
Test_R2      0
dtype: int64

Jumlah kombinasi Pollutant-Horizon:
48


In [61]:
# Ranking model berdasarkan Validation MAE
# Semakin kecil MAE = semakin baik

ranking_val = (
    results_all
    .sort_values(
        by=["Pollutant", "Horizon", "Val_MAE"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

display(ranking_val)

,Pollutant,Horizon,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
0,CO,1H,Ridge,94.290586,159.999809,0.475728,131.490764,210.144449,0.532923
1,CO,1H,Persistence,98.155647,167.346916,0.426474,133.809374,222.390026,0.476901
2,CO,1H,HistGradientBoosting,105.254352,179.336383,0.341351,169.862605,280.771148,0.166208
3,CO,1H,Random Forest,109.707122,184.864405,0.300119,180.816075,288.986645,0.116699
4,CO,2H,Ridge,120.767879,199.298781,0.186389,171.813010,268.727330,0.236994
...,...,...,...,...,...,...,...,...,...
187,SO2,72H,Persistence,2.112093,4.691686,-0.807500,2.458523,4.785185,-0.858550
188,SO2,7H,Random Forest,1.357384,3.540031,-0.069537,1.563704,3.441697,-0.092165
189,SO2,7H,HistGradientBoosting,1.416945,3.593600,-0.102152,1.593882,3.441115,-0.091796
190,SO2,7H,Ridge,1.563019,3.563711,-0.083894,1.774830,3.422863,-0.080245


In [62]:
# ============================================================
# Model terbaik berdasarkan Validation MAE
# ============================================================

best_models_val = (
    results_all
    .sort_values(
        by=["Pollutant", "Horizon", "Val_MAE"],
        ascending=[True, True, True]
    )
    .groupby(["Pollutant", "Horizon"], as_index=False)
    .first()
)

# Pilih kolom penting saja
best_models_val = best_models_val[
    [
        "Pollutant",
        "Horizon",
        "Model",
        "Val_MAE",
        "Val_RMSE",
        "Val_R2",
        "Test_MAE",
        "Test_RMSE",
        "Test_R2"
    ]
]

display(best_models_val)

,Pollutant,Horizon,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
0,CO,1H,Ridge,94.290586,159.999809,0.475728,131.490764,210.144449,0.532923
1,CO,2H,Ridge,120.767879,199.298781,0.186389,171.813010,268.727330,0.236994
2,CO,3H,Ridge,130.238619,213.363654,0.067239,193.844751,295.052819,0.081723
3,CO,4H,Ridge,132.043225,214.730826,0.054611,204.444295,306.686983,0.009744
4,CO,5H,Ridge,134.329740,219.963852,0.008219,216.912239,320.435408,-0.079365
5,CO,6H,Ridge,137.820134,223.732300,-0.025241,221.714198,330.866462,-0.149506
6,CO,72H,Random Forest,129.421158,203.698341,0.171509,200.785217,297.754518,0.136701
7,CO,7H,HistGradientBoosting,140.761384,221.049251,-0.000634,218.072782,333.886468,-0.168952
8,NO2,1H,HistGradientBoosting,1.809301,2.341430,0.929815,2.156637,2.865583,0.905958
9,NO2,2H,Random Forest,2.358096,3.042640,0.881478,2.800446,3.653451,0.847497


In [63]:
# Mengurutkan horizon secara numerik

horizon_order = ["1H", "2H", "3H", "4H", "5H", "6H", "7H", "72H"]

best_models_val["Horizon"] = pd.Categorical(
    best_models_val["Horizon"],
    categories=horizon_order,
    ordered=True
)

best_models_val = (
    best_models_val
    .sort_values(["Pollutant", "Horizon"])
    .reset_index(drop=True)
)

display(best_models_val)

,Pollutant,Horizon,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
0,CO,1H,Ridge,94.290586,159.999809,0.475728,131.490764,210.144449,0.532923
1,CO,2H,Ridge,120.767879,199.298781,0.186389,171.813010,268.727330,0.236994
2,CO,3H,Ridge,130.238619,213.363654,0.067239,193.844751,295.052819,0.081723
3,CO,4H,Ridge,132.043225,214.730826,0.054611,204.444295,306.686983,0.009744
4,CO,5H,Ridge,134.329740,219.963852,0.008219,216.912239,320.435408,-0.079365
5,CO,6H,Ridge,137.820134,223.732300,-0.025241,221.714198,330.866462,-0.149506
6,CO,7H,HistGradientBoosting,140.761384,221.049251,-0.000634,218.072782,333.886468,-0.168952
7,CO,72H,Random Forest,129.421158,203.698341,0.171509,200.785217,297.754518,0.136701
8,NO2,1H,HistGradientBoosting,1.809301,2.341430,0.929815,2.156637,2.865583,0.905958
9,NO2,2H,Random Forest,2.358096,3.042640,0.881478,2.800446,3.653451,0.847497


In [64]:
# Frekuensi model yang terpilih berdasarkan Validation MAE

model_selection_count = (
    best_models_val["Model"]
    .value_counts()
    .rename_axis("Model")
    .reset_index(name="Jumlah Terpilih")
)

display(model_selection_count)

,Model,Jumlah Terpilih
0,Random Forest,30
1,Ridge,9
2,HistGradientBoosting,9


In [65]:
# Simpan hasil pemilihan model
# Berdasarkan Validation MAE

final_model_selection = best_models_val.copy()

# Urutkan berdasarkan pollutant dan horizon
final_model_selection = (
    final_model_selection
    .sort_values(["Pollutant", "Horizon"])
    .reset_index(drop=True)
)

print("Jumlah model final:", len(final_model_selection))

display(final_model_selection)

Jumlah model final: 48


,Pollutant,Horizon,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
0,CO,1H,Ridge,94.290586,159.999809,0.475728,131.490764,210.144449,0.532923
1,CO,2H,Ridge,120.767879,199.298781,0.186389,171.813010,268.727330,0.236994
2,CO,3H,Ridge,130.238619,213.363654,0.067239,193.844751,295.052819,0.081723
3,CO,4H,Ridge,132.043225,214.730826,0.054611,204.444295,306.686983,0.009744
4,CO,5H,Ridge,134.329740,219.963852,0.008219,216.912239,320.435408,-0.079365
5,CO,6H,Ridge,137.820134,223.732300,-0.025241,221.714198,330.866462,-0.149506
6,CO,7H,HistGradientBoosting,140.761384,221.049251,-0.000634,218.072782,333.886468,-0.168952
7,CO,72H,Random Forest,129.421158,203.698341,0.171509,200.785217,297.754518,0.136701
8,NO2,1H,HistGradientBoosting,1.809301,2.341430,0.929815,2.156637,2.865583,0.905958
9,NO2,2H,Random Forest,2.358096,3.042640,0.881478,2.800446,3.653451,0.847497


In [66]:
# ============================================================
# Fungsi training final
# Train + Validation -> Final Model
# Test tetap disimpan untuk evaluasi akhir
# ============================================================

def train_final_model(pollutant, horizon, model_name):
    """
    Melatih model final menggunakan gabungan Train + Validation.
    Test tidak digunakan dalam proses fitting.
    """

    target_col = f"{pollutant}_target_{horizon.replace('H', '')}h"

    # Bangun dataset
    X, y = build_dataset(target_col)

    # Time split
    (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    ) = time_split(X, y)

    # Gabungkan Train + Validation
    X_train_final = pd.concat([X_train, X_val])
    y_train_final = pd.concat([y_train, y_val])

    # Pilih model sesuai hasil eksperimen
    if model_name == "Ridge":

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ])

    elif model_name == "Random Forest":

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            max_features="sqrt"
        )

    elif model_name == "HistGradientBoosting":

        model = HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=42
        )

    else:
        raise ValueError(
            f"Model '{model_name}' tidak didukung untuk training final."
        )

    # Training final
    model.fit(X_train_final, y_train_final)

    return model

In [67]:
# ============================================================
# TRAINING 48 MODEL FINAL
# Train + Validation
# ============================================================

final_models = {}

for i, row in final_model_selection.iterrows():

    pollutant = row["Pollutant"]
    horizon = row["Horizon"]
    model_name = row["Model"]

    print(
        f"[{i+1}/48] "
        f"{pollutant} - {horizon} - {model_name}",
        flush=True
    )

    model = train_final_model(
        pollutant=pollutant,
        horizon=horizon,
        model_name=model_name
    )

    # Key model: contoh "PM25_1H"
    model_key = f"{pollutant}_{horizon}"

    final_models[model_key] = model

print("\n========================================")
print("TRAINING MODEL FINAL SELESAI")
print("========================================")
print("Jumlah model:", len(final_models))
print("Seharusnya  :", 48)

[1/48] CO - 1H - Ridge
[2/48] CO - 2H - Ridge
[3/48] CO - 3H - Ridge
[4/48] CO - 4H - Ridge
[5/48] CO - 5H - Ridge
[6/48] CO - 6H - Ridge
[7/48] CO - 7H - HistGradientBoosting
[8/48] CO - 72H - Random Forest
[9/48] NO2 - 1H - HistGradientBoosting
[10/48] NO2 - 2H - Random Forest
[11/48] NO2 - 3H - Random Forest
[12/48] NO2 - 4H - Random Forest
[13/48] NO2 - 5H - Random Forest
[14/48] NO2 - 6H - Random Forest
[15/48] NO2 - 7H - Random Forest
[16/48] NO2 - 72H - Random Forest
[17/48] O3 - 1H - HistGradientBoosting
[18/48] O3 - 2H - HistGradientBoosting
[19/48] O3 - 3H - HistGradientBoosting
[20/48] O3 - 4H - HistGradientBoosting
[21/48] O3 - 5H - HistGradientBoosting
[22/48] O3 - 6H - HistGradientBoosting
[23/48] O3 - 7H - HistGradientBoosting
[24/48] O3 - 72H - Random Forest
[25/48] PM10 - 1H - Random Forest
[26/48] PM10 - 2H - Random Forest
[27/48] PM10 - 3H - Random Forest
[28/48] PM10 - 4H - Random Forest
[29/48] PM10 - 5H - Random Forest
[30/48] PM10 - 6H - Random Forest
[31/48] PM1

In [68]:
# ============================================================
# EVALUASI FINAL 48 MODEL PADA TEST SET
# ============================================================

final_test_results = []

for i, row in final_model_selection.iterrows():

    pollutant = row["Pollutant"]
    horizon = row["Horizon"]
    model_name = row["Model"]

    target_col = f"{pollutant}_target_{horizon.replace('H', '')}h"

    # Bangun dataset
    X, y = build_dataset(target_col)

    # Time split
    (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    ) = time_split(X, y)

    # Ambil model final
    model_key = f"{pollutant}_{horizon}"
    model = final_models[model_key]

    # Prediksi TEST
    y_pred_test = model.predict(X_test)

    # Evaluasi
    final_test_results.append({
        "Pollutant": pollutant,
        "Horizon": horizon,
        "Model": model_name,
        "Test_MAE": mean_absolute_error(y_test, y_pred_test),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Test_R2": r2_score(y_test, y_pred_test)
    })

final_test_results = pd.DataFrame(final_test_results)

print("Jumlah model dievaluasi:", len(final_test_results))

display(final_test_results)

Jumlah model dievaluasi: 48


,Pollutant,Horizon,Model,Test_MAE,Test_RMSE,Test_R2
0,CO,1H,Ridge,119.981716,197.862391,0.585925
1,CO,2H,Ridge,158.136564,250.094460,0.339135
2,CO,3H,Ridge,179.356054,274.512025,0.205129
3,CO,4H,Ridge,189.702149,287.530689,0.129587
4,CO,5H,Ridge,198.811057,299.752207,0.055478
5,CO,6H,Ridge,203.372361,303.354010,0.033715
6,CO,7H,HistGradientBoosting,178.134753,288.659156,0.126285
7,CO,72H,Random Forest,190.314781,297.859137,0.136094
8,NO2,1H,HistGradientBoosting,2.077721,2.820457,0.908897
9,NO2,2H,Random Forest,2.737138,3.623704,0.849970


In [69]:
# ============================================================
# Cek data terakhir yang akan digunakan untuk forecasting
# ============================================================

latest_features = df_model[feature_cols].iloc[[-1]]

print("Timestamp terakhir:", latest_features.index[0])
print("Jumlah fitur:", latest_features.shape[1])
print("Jumlah missing:", latest_features.isna().sum().sum())

print("\nFeature yang masih missing:")
print(latest_features.columns[latest_features.isna().any()].tolist())

display(latest_features)

Timestamp terakhir: 2026-09-21 17:00:00
Jumlah fitur: 58
Jumlah missing: 0

Feature yang masih missing:
[]


,NO2,SO2,PM25,PM10,CO,O3,TMP,RAIN,HUM,WS,...,SO2_roll_mean_3,SO2_roll_mean_6,NO2_roll_mean_3,NO2_roll_mean_6,WD_sin,WD_cos,hour,day_of_week,hour_sin,hour_cos
2026-09-21 17:00:00,11.193537,1.771957,55.423392,99.176302,639.042579,38.541698,34.011652,0.0,66.491052,1.391176,...,2.001485,2.682455,23.965787,26.121503,0.811716,-0.584052,17,0,-0.965926,-0.258819


In [70]:
# ============================================================
# FORECAST 48 TARGET DARI TIMESTAMP TERAKHIR
# ============================================================

latest_features = df_model[feature_cols].iloc[[-1]]
latest_timestamp = latest_features.index[0]

forecast_results = []

for i, row in final_model_selection.iterrows():

    pollutant = row["Pollutant"]
    horizon = row["Horizon"]
    model_name = row["Model"]

    # Ambil model final
    model_key = f"{pollutant}_{horizon}"
    model = final_models[model_key]

    # Prediksi
    prediction = model.predict(latest_features)[0]

    # Konversi horizon string -> jam
    horizon_hours = int(horizon.replace("H", ""))

    # Waktu forecast
    forecast_time = latest_timestamp + pd.Timedelta(
        hours=horizon_hours
    )

    forecast_results.append({
        "Forecast_From": latest_timestamp,
        "Forecast_Time": forecast_time,
        "Pollutant": pollutant,
        "Horizon": horizon,
        "Model": model_name,
        "Prediction": prediction
    })

forecast_results = pd.DataFrame(forecast_results)

# Urutkan berdasarkan pollutant dan waktu forecast
forecast_results = (
    forecast_results
    .sort_values(["Pollutant", "Forecast_Time"])
    .reset_index(drop=True)
)

print("Jumlah forecast:", len(forecast_results))

display(forecast_results)

Jumlah forecast: 48


,Forecast_From,Forecast_Time,Pollutant,Horizon,Model,Prediction
0,2026-09-21 17:00:00,2026-09-21 18:00:00,CO,1H,Ridge,528.216856
1,2026-09-21 17:00:00,2026-09-21 19:00:00,CO,2H,Ridge,475.293336
2,2026-09-21 17:00:00,2026-09-21 20:00:00,CO,3H,Ridge,409.550294
3,2026-09-21 17:00:00,2026-09-21 21:00:00,CO,4H,Ridge,388.007048
4,2026-09-21 17:00:00,2026-09-21 22:00:00,CO,5H,Ridge,385.913335
5,2026-09-21 17:00:00,2026-09-21 23:00:00,CO,6H,Ridge,322.242818
6,2026-09-21 17:00:00,2026-09-22 00:00:00,CO,7H,HistGradientBoosting,499.836900
7,2026-09-21 17:00:00,2026-09-24 17:00:00,CO,72H,Random Forest,614.619164
8,2026-09-21 17:00:00,2026-09-21 18:00:00,NO2,1H,HistGradientBoosting,9.604992
9,2026-09-21 17:00:00,2026-09-21 19:00:00,NO2,2H,Random Forest,10.098872


In [71]:
# ============================================================
# SANITY CHECK HASIL FORECAST
# ============================================================

print("=== CHECK NILAI NEGATIF ===")

negative_forecasts = forecast_results[
    forecast_results["Prediction"] < 0
]

print("Jumlah forecast negatif:", len(negative_forecasts))

if len(negative_forecasts) > 0:
    display(negative_forecasts)
else:
    print("Tidak ada forecast negatif.")


print("\n=== RANGE FORECAST PER POLUTAN ===")

forecast_summary = (
    forecast_results
    .groupby("Pollutant")["Prediction"]
    .agg(["min", "max", "mean"])
    .reset_index()
)

display(forecast_summary)


print("\n=== CURRENT VALUE vs FORECAST ===")

current_values = df_model[pollutant_cols].iloc[-1]

print("Timestamp:", df_model.index[-1])

for pollutant in pollutant_cols:
    print(
        f"{pollutant:5s} current = "
        f"{current_values[pollutant]:.3f}"
    )

=== CHECK NILAI NEGATIF ===
Jumlah forecast negatif: 0
Tidak ada forecast negatif.

=== RANGE FORECAST PER POLUTAN ===


,Pollutant,min,max,mean
0,CO,322.242818,614.619164,452.959969
1,NO2,9.604992,17.421166,12.914643
2,O3,27.757588,46.620415,41.253798
3,PM10,67.717462,93.763524,77.153119
4,PM25,36.463158,48.343299,41.559875
5,SO2,0.207861,3.658230,2.312230



=== CURRENT VALUE vs FORECAST ===
Timestamp: 2026-09-21 17:00:00
PM25  current = 55.423
PM10  current = 99.176
CO    current = 639.043
O3    current = 38.542
SO2   current = 1.772
NO2   current = 11.194


In [72]:
# ============================================================
# TABEL FORECAST FINAL
# 9 TIMESTAMP × 6 POLUTAN
# ============================================================

forecast_table = (
    forecast_results
    .pivot(
        index="Forecast_Time",
        columns="Pollutant",
        values="Prediction"
    )
    .reset_index()
)

# Pastikan urutan kolom sesuai kebutuhan website
forecast_table = forecast_table[
    ["Forecast_Time", "PM25", "PM10", "CO", "O3", "SO2", "NO2"]
]

# Urutkan berdasarkan waktu forecast
forecast_table = (
    forecast_table
    .sort_values("Forecast_Time")
    .reset_index(drop=True)
)

print("Jumlah timestamp forecast:", len(forecast_table))
display(forecast_table)

Jumlah timestamp forecast: 8


Pollutant,Forecast_Time,PM25,PM10,CO,O3,SO2,NO2
0,2026-09-21 18:00:00,48.343299,87.886137,528.216856,46.566152,0.207861,9.604992
1,2026-09-21 19:00:00,41.726818,78.933267,475.293336,44.819247,2.831469,10.098872
2,2026-09-21 20:00:00,40.178135,75.234335,409.550294,44.130676,3.036444,11.059954
3,2026-09-21 21:00:00,37.032047,72.866769,388.007048,41.075233,2.796492,13.064905
4,2026-09-21 22:00:00,45.715024,71.973654,385.913335,46.620415,2.064205,13.656853
5,2026-09-21 23:00:00,37.367075,67.717462,322.242818,42.481612,1.994611,14.584112
6,2026-09-22 00:00:00,36.463158,68.849803,499.836900,36.579460,1.908531,17.421166
7,2026-09-24 17:00:00,45.653446,93.763524,614.619164,27.757588,3.658230,13.826290


In [73]:
# ============================================================
# SIMPAN 48 MODEL FINAL + METADATA
# ============================================================

import os
import joblib

# Folder penyimpanan model
model_dir = "../models/forecast_ispu"
os.makedirs(model_dir, exist_ok=True)

# Simpan masing-masing model
for model_key, model in final_models.items():
    model_path = os.path.join(
        model_dir,
        f"{model_key}.joblib"
    )
    
    joblib.dump(model, model_path)

# Simpan metadata pemilihan model
metadata_path = os.path.join(
    model_dir,
    "model_selection.csv"
)

final_model_selection.to_csv(
    metadata_path,
    index=False
)

print("=== MODEL SAVED ===")
print("Jumlah model:", len(final_models))
print("Folder:", model_dir)
print("Metadata:", metadata_path)

=== MODEL SAVED ===
Jumlah model: 48
Folder: ../models/forecast_ispu
Metadata: ../models/forecast_ispu\model_selection.csv


In [75]:
# ============================================================
# SIMPAN OUTPUT FORECAST DI FOLDER TERPISAH
# ============================================================

import os

forecast_dir = "../outputs/forecast"
os.makedirs(forecast_dir, exist_ok=True)

# Output detail: 48 prediksi
forecast_results.to_csv(
    os.path.join(forecast_dir, "forecast_results.csv"),
    index=False
)

# Output utama untuk website:
# 8 timestamp × 6 pollutant
forecast_table.to_csv(
    os.path.join(forecast_dir, "forecast_table.csv"),
    index=False
)

print("=== FORECAST OUTPUT ===")
print("Folder:", forecast_dir)
print("forecast_results.csv :", len(forecast_results), "rows")
print("forecast_table.csv   :", len(forecast_table), "rows")

=== FORECAST OUTPUT ===
Folder: ../outputs/forecast
forecast_results.csv : 48 rows
forecast_table.csv   : 8 rows
